# 04 - Fine tuning de ResNet18

Aplicamos transfer learning: empezamos con todo congelado salvo la cabeza,
luego descongelamos las ultimas capas y entrenamos con LR discriminativos
(uno para la cabeza y otro mas pequenio para el backbone descongelado).

Etapas:
1. Setup, config y dataloaders.
2. Construccion del modelo y politica de freezing.
3. Entrenamiento con W&B opcional y validacion por epoch.
4. Evaluacion final en validacion, checkpoint y registro final.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.utils.config import load_yaml_config
from src.utils.reproducibility import set_global_seed

import torch
from torch import nn, optim

from src.models.transfer import build_resnet18_finetune
from src.training.engine import evaluate_classification, train_one_epoch
from src.utils.wandb_utils import finish_wandb_run, init_wandb_run

CONFIG_PATH = PROJECT_ROOT / "configs" / "finetune.yaml"
config = load_yaml_config(CONFIG_PATH)
set_global_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else config["device"])
print("Device:", device)
config

## Data loaders

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.data.dataset import load_imagefolder_datasets

data_root = PROJECT_ROOT / config["data"]["root_dir"]
image_size = config["data"]["image_size"]
batch_size = config["data"]["batch_size"]
num_workers = config["data"].get("num_workers", 0)

train_dataset, val_dataset = load_imagefolder_datasets(
    root_dir=data_root,
    train_subdir=config["data"].get("train_subdir", "train"),
    val_subdir=config["data"].get("val_subdir", "val"),
    image_size=image_size,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

class_names = train_dataset.classes
num_classes = len(class_names)
print({"train": len(train_dataset), "val": len(val_dataset), "num_classes": num_classes})


## Modelo preentrenado y optimizador con LR discriminativos

In [ ]:
model = build_resnet18_finetune(
    num_classes=num_classes,
    freeze_backbone=config["model"].get("freeze_backbone", True),
    unfreeze_last_n_layers=config["model"].get("unfreeze_last_n_layers", 1),
).to(device)

head_params = [p for n, p in model.named_parameters() if n.startswith("fc.") and p.requires_grad]
backbone_params = [p for n, p in model.named_parameters() if not n.startswith("fc.") and p.requires_grad]

param_groups = [{"params": head_params, "lr": config["training"]["learning_rate_head"]}]
if backbone_params:
    param_groups.append({"params": backbone_params, "lr": config["training"]["learning_rate_backbone"]})

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(param_groups, weight_decay=config["training"].get("weight_decay", 0.0))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

## Loop de fine tuning

In [ ]:
run = init_wandb_run(
    config=config,
    enabled=config["tracking"].get("use_wandb", False),
    project=config["tracking"]["project"],
    run_name=config["tracking"].get("run_name", config["experiment_name"]),
    tags=config["tracking"].get("tags"),
)

history = []
for epoch in range(1, config["training"]["epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate_classification(model, val_loader, criterion, device)
    entry = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
    }
    history.append(entry)
    if run is not None:
        run.log(entry)
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

## Evaluacion final en validacion y guardado del checkpoint

In [ ]:
final_val_loss, final_val_acc = evaluate_classification(model, val_loader, criterion, device)
print({"final_val_loss": final_val_loss, "final_val_accuracy": final_val_acc})

output_dir = PROJECT_ROOT / config["output"]["artifacts_dir"] / "finetune"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / config["output"]["checkpoint_name"]
torch.save({
    "state_dict": model.state_dict(),
    "model_type": "resnet18_finetune",
    "class_names": class_names,
    "image_size": image_size,
    "backbone": config["model"].get("backbone", "resnet18"),
    "freeze_backbone": config["model"].get("freeze_backbone", True),
    "unfreeze_last_n_layers": config["model"].get("unfreeze_last_n_layers", 1),
}, checkpoint_path)
print("Saved checkpoint to", checkpoint_path)

if run is not None:
    run.log({"final_val_loss": final_val_loss, "final_val_accuracy": final_val_acc})
    finish_wandb_run(run)